# Normalización automática de comentarios

Aplica las reglas automatizables del protocolo de normalización del proyecto y agrega la columna `text_normalized` al CSV descargado.

**Qué hace:** usuarios → @USER; emojis → «[3 emoticonos de risa]»; comentarios casi todo en mayúsculas → minúsculas (conserva siglas y mayúsculas enfáticas dentro de texto normal); risas → ja / jaja / jajaja; puntos seguidos → «…»; abreviaciones (q, pq, xq, tb…) → forma plena; nombres propios, países y siglas de una lista; signos ¿ y ¡ faltantes; mayúscula al inicio de oración.

**Qué no hace (requiere revisión humana):** agregar tildes en general, corregir ortografía, marcar [sic] y decidir «porque» frente a «por qué». La columna `revisar` señala algunos de estos casos.

In [ ]:
!pip -q install emoji

## 1. Cargar el CSV

Sube el archivo con el ícono de carpeta de la izquierda o déjalo en Drive y cambia la ruta.

In [ ]:
ENTRADA = "comentarios_youtube.csv"
SALIDA = "comentarios_youtube_normalizados.csv"
COLUMNA_TEXTO = "comment_text"   # texto con autores anonimizados (@USER)

import pandas as pd
df = pd.read_csv(ENTRADA)
print(len(df), "comentarios")

## 2. Reglas de normalización

Las listas `ABREVIACIONES`, `NOMBRES` y `SIGLAS` se pueden ampliar sin tocar el resto.

In [ ]:
import re
import emoji

# Nombres del protocolo de normalización (se usan antes que los nombres automáticos)
EMOJIS_PROTOCOLO = {
    "😂": "risa", "🤣": "risa", "👍": "pulgar arriba", "🙏": "manos juntas", "👏": "aplausos",
    "💯": "“100”", "💩": "excremento", "🤡": "rostro de payaso", "🎉": "corneta de fiestas",
    "🗾": "mapa de Japón", "🗺": "mapa mundi", "🔱": "tridente", "⚓": "ancla", "🐽": "nariz de cerdo",
    "©": "símbolo de copyright", "🎷": "saxofón", "🎸": "guitarra", "➕": "signo más", "🆗": "OK",
    "👌": "mano haciendo OK", "🕊": "paloma de la paz", "🔆": "sol brillante",
    "🔊": "altavoz a todo volumen", "🛎": "timbre de recepción", "😮": "sorpresa",
}

# Abreviaciones legibles -> forma plena (editable)
ABREVIACIONES = {
    "x que": "porque", "osea": "o sea", "esq": "es que", "asi": "así", "q": "que", "k": "que", "xq": "porque", "pq": "porque", "porq": "porque", "x q": "porque",
    "x": "por", "tb": "también", "tbn": "también", "tmb": "también", "dd": "dónde",
    "dps": "después", "pa": "para", "xfa": "por favor", "xfavor": "por favor", "tqm": "te quiero mucho",
}

# Nombres propios, países y siglas en su forma normalizada (editable)
NOMBRES = ["Kast", "Boric", "Piñera", "Bachelet", "Kaiser", "Jara", "Matthei", "Milei", "Trump",
           "Chile", "Perú", "Bolivia", "Argentina", "Venezuela", "Irán", "Israel", "Estados Unidos",
           "Santiago", "Antofagasta", "Concepción", "Valparaíso", "Biobío", "Japón", "China"]
SIGLAS = ["PDI", "CHV", "TVN", "AFP", "MEPCO", "ENAP", "IVA", "FMI", "RM", "BCI", "UF", "PGU", "EE. UU.", "USD"]


def _sin_tildes(s):
    return s.translate(str.maketrans("áéíóúÁÉÍÓÚ", "aeiouAEIOU"))


def _emoji_a_texto(e):
    base = e.replace("️", "")
    base = re.sub("[\U0001F3FB-\U0001F3FF]", "", base)  # quita el tono de piel
    if base in EMOJIS_PROTOCOLO:
        return EMOJIS_PROTOCOLO[base]
    nombre = emoji.demojize(base, language="es").strip(":").replace("_", " ")
    if nombre.startswith("bandera "):
        nombre = "bandera de " + nombre[len("bandera "):].title()
    return nombre


def emojis(texto):
    """Agrupa emojis iguales seguidos: 😂😂😂 -> [3 emoticonos de risa]."""
    partes = emoji.emoji_list(texto)
    if not partes:
        return texto
    out, i, idx = [], 0, 0
    while idx < len(partes):
        p = partes[idx]
        out.append(texto[i:p["match_start"]])
        nombre = _emoji_a_texto(p["emoji"])
        n, fin = 1, p["match_end"]
        while idx + 1 < len(partes) and _emoji_a_texto(partes[idx + 1]["emoji"]) == nombre \
                and texto[fin:partes[idx + 1]["match_start"]].strip() == "":
            idx += 1; n += 1; fin = partes[idx]["match_end"]
        out.append(f" [{n} {'emoticono' if n == 1 else 'emoticonos'} de {nombre}] ")
        i = fin; idx += 1
    out.append(texto[i:])
    return "".join(out)


def risas(texto):
    def rep(m):
        n = len(re.findall(r"ja", m.group(0), re.I))
        return "jajaja" if n >= 3 else ("jaja" if n == 2 else "ja")
    return re.sub(r"(?<![a-záéíóúñ])j*(?:ja)+j*a*(?![a-záéíóúñ])", rep, texto, flags=re.I)


def mayusculas(texto):
    """Si el comentario está casi todo en mayúsculas, lo pasa a minúsculas (se conservan siglas)."""
    letras = [c for c in texto if c.isalpha()]
    if len(letras) >= 8 and sum(c.isupper() for c in letras) / len(letras) > 0.6:
        return texto.lower()
    return texto


def alargamientos(texto):
    """Reduce letras repetidas 3 o más veces: «Nooo» -> «No», «siiii» -> «si»."""
    return re.sub(r"([a-záéíóúñ])\1{2,}", r"\1", texto, flags=re.I)


def abreviaciones(texto):
    for k in sorted(ABREVIACIONES, key=len, reverse=True):
        texto = re.sub(rf"(?<![\wáéíóúñ]){re.escape(k)}(?![\wáéíóúñ])", ABREVIACIONES[k], texto, flags=re.I)
    return texto


def nombres_y_siglas(texto):
    for n in NOMBRES + SIGLAS:
        patron = re.escape(_sin_tildes(n)).replace("a", "[aá]").replace("e", "[eé]").replace("i", "[ií]") \
                                          .replace("o", "[oó]").replace("u", "[uú]")
        texto = re.sub(rf"(?<![\wáéíóúñ]){patron}(?![\wáéíóúñ])", n, texto, flags=re.I)
    return texto


def puntuacion(texto):
    texto = re.sub(r"\.{2,}(\s*\.)*|…+", "…", texto)         # puntos seguidos -> …
    texto = re.sub(r"…(?=[^\s,.;:!?)\]])", "… ", texto)       # espacio después de …
    texto = re.sub(r"([!?¡¿])\1+", r"\1", texto)              # ??? -> ?
    texto = re.sub(r"\s+([,.;:!?…])", r"\1", texto)           # sin espacio antes de signo
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r" *\n *", "\n", texto)
    return texto.strip()


def signos_apertura(texto):
    """Agrega ¿ o ¡ al inicio del enunciado cuando falta: 'Y la zanja?' -> '¿Y la zanja?'."""
    trozos = re.split(r"(?<=[.!?…\n])(\s+)", texto)
    res = []
    for t in trozos:
        if t.strip():
            if re.search(r"(\?!|!\?)$", t) and "¿" not in t and "¡" not in t:
                t = "¡¿" + t[:-2] + "?!"
            elif t.endswith("?") and "¿" not in t:
                t = "¿" + t
            elif t.endswith("!") and "¡" not in t:
                t = "¡" + t
        res.append(t)
    return "".join(res)


def mayuscula_inicial(texto):
    texto = re.sub(r"^([¿¡\"“«(]*)([a-záéíóúñ])", lambda m: m.group(1) + m.group(2).upper(), texto)
    return re.sub(r"([.!?…]\s+[¿¡\"“«(]*)([a-záéíóúñ])", lambda m: m.group(1) + m.group(2).upper(), texto)


def normalizar(texto):
    if not isinstance(texto, str):
        return texto
    t = re.sub(r"@[\w.\-]+", "@USER", texto)
    t = emojis(t)
    t = mayusculas(t)
    t = risas(t)
    t = alargamientos(t)
    t = abreviaciones(t)
    t = nombres_y_siglas(t)
    t = puntuacion(t)
    t = signos_apertura(t)
    t = mayuscula_inicial(t)
    t = re.sub(r"@user\b", "@USER", t, flags=re.I)   # las minúsculas no deben tocar @USER
    return t


def requiere_revision(texto):
    """Marca lo que el protocolo pide y no se puede hacer bien de forma automática."""
    motivos = []
    if re.search(r"(?<![\wáéíóúñ])(porque)(?![\wáéíóúñ])[^.?!]*\?", texto if isinstance(texto, str) else ""):
        motivos.append("¿«porque» o «por qué»?")
    return "; ".join(motivos)


## 3. Normalizar

In [ ]:
df["text_normalized"] = df[COLUMNA_TEXTO].apply(normalizar)
df["revisar"] = df["text_normalized"].apply(requiere_revision)
print("Comentarios marcados para revisar:", (df["revisar"] != "").sum())

## 4. Revisar una muestra

Compara el original con el normalizado en 20 comentarios al azar.

In [ ]:
pd.set_option("display.max_colwidth", 200)
df.sample(20, random_state=1)[[COLUMNA_TEXTO, "text_normalized", "revisar"]]

## 5. Guardar

In [ ]:
df.to_csv(SALIDA, index=False, encoding="utf-8-sig")
from google.colab import files
files.download(SALIDA)